# Encounters Pipeline

Run cells top-to-bottom. Uses parquet caching for fast reloads.


In [ ]:
from pathlib import Path
import sys
import subprocess

try:
    import pandas as pd
    from tabulate import tabulate
except ModuleNotFoundError as e:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pandas', 'tabulate'])
    import pandas as pd
    from tabulate import tabulate

def show(df, title=None, n=20):
    if title:
        print(f"\n{'='*60}")
        print(f"  {title}")
        print('='*60)
    print(tabulate(df.head(n), headers='keys', tablefmt='pretty', showindex=False))
    print(f"\n(Showing {n} of {len(df):,} rows)")


In [ ]:
csv_path = Path('encounters.csv')
if not csv_path.exists():
    csv_path = Path('/Users/calvin/Documents/datafest/encounters.csv')
if not csv_path.exists():
    raise FileNotFoundError('encounters.csv not found')

cache_path = csv_path.with_name('encounters_cache.parquet')

if cache_path.exists() and cache_path.stat().st_mtime >= csv_path.stat().st_mtime:
    df = pd.read_parquet(cache_path)
    print(f"Loaded from cache: {cache_path.name}")
else:
    try:
        df = pd.read_csv(csv_path, engine='pyarrow', low_memory=False)
        engine = 'pyarrow'
    except Exception:
        df = pd.read_csv(csv_path, engine='c', low_memory=False)
        engine = 'c'
    df.to_parquet(cache_path, index=False)
    print(f"Loaded CSV ({engine}) and cached: {cache_path.name}")

print(f"\nDataset: {len(df):,} rows x {df.shape[1]} columns")
print(f"Columns: {', '.join(df.columns.tolist())}")


In [ ]:
def to_num(s):
    return pd.to_numeric(s, errors='coerce')


In [ ]:
# Step 1: Sort by Admit Date/Time (Year, Month, Day, Hour, Minute) ascending
step1 = df.copy()
for c in ['AdmitYear', 'AdmitMonth', 'AdmitDay', 'AdmitHour', 'AdmitMinute']:
    step1[f'__{c}_num'] = to_num(step1[c])

step1 = step1.sort_values(
    ['__AdmitYear_num', '__AdmitMonth_num', '__AdmitDay_num', '__AdmitHour_num', '__AdmitMinute_num'],
    ascending=True,
    na_position='last',
    kind='stable',
)

show(step1[['AdmitYear','AdmitMonth','AdmitDay','AdmitHour','AdmitMinute']], 'Step 1: Admit Date/Time (Ascending)')


In [ ]:
# Step 2: Group by AdmissionSource
admission_source_sections = {k: g.copy() for k, g in step1.groupby('AdmissionSource', dropna=False)}
admission_source_summary = (
    step1.groupby('AdmissionSource', dropna=False)
    .size()
    .reset_index(name='RowCount')
    .sort_values('RowCount', ascending=False)
)

show(admission_source_summary, 'Step 2: Admission Source Summary')


In [ ]:
# Step 3: Group by AdmissionType
admission_type_sections = {k: g.copy() for k, g in step1.groupby('AdmissionType', dropna=False)}
admission_type_summary = (
    step1.groupby('AdmissionType', dropna=False)
    .size()
    .reset_index(name='RowCount')
    .sort_values('RowCount', ascending=False)
)

show(admission_type_summary, 'Step 3: Admission Type Summary')


In [ ]:
# Step 4: Sort by Discharge Date/Time ascending
step4 = step1.copy()
step4['__DischargeInstant_dt'] = pd.to_datetime(step4['DischargeInstant'], errors='coerce')
step4 = step4.sort_values('__DischargeInstant_dt', ascending=True, na_position='last', kind='stable')

show(step4[['DischargeInstant']], 'Step 4: Discharge Date/Time (Ascending)')


In [ ]:
# Step 5: Sort by AttendingProviderDurableKey ascending
step5 = step4.copy()
step5['__AttendingProviderDurableKey_num'] = to_num(step5['AttendingProviderDurableKey'])
step5 = step5.sort_values('__AttendingProviderDurableKey_num', ascending=True, na_position='last', kind='stable')

show(step5[['AttendingProviderDurableKey']], 'Step 5: Attending Provider (Ascending)')


In [ ]:
# Step 6: Sort by DischargeProviderDurableKey ascending
step6 = step5.copy()
step6['__DischargeProviderDurableKey_num'] = to_num(step6['DischargeProviderDurableKey'])
step6 = step6.sort_values('__DischargeProviderDurableKey_num', ascending=True, na_position='last', kind='stable')

show(step6[['DischargeProviderDurableKey']], 'Step 6: Discharge Provider (Ascending)')


In [ ]:
# Step 7: Sort by DepartmentKey ascending
step7 = step6.copy()
step7['__DepartmentKey_num'] = to_num(step7['DepartmentKey'])
step7 = step7.sort_values('__DepartmentKey_num', ascending=True, na_position='last', kind='stable')

show(step7[['DepartmentKey']], 'Step 7: Department (Ascending)')


In [ ]:
# Step 8: Sort by PrimaryDiagnosisKey ascending
final_df = step7.copy()
final_df['__PrimaryDiagnosisKey_num'] = to_num(final_df['PrimaryDiagnosisKey'])
final_df = final_df.sort_values('__PrimaryDiagnosisKey_num', ascending=True, na_position='last', kind='stable')

show(final_df[['PrimaryDiagnosisKey']], 'Step 8: Primary Diagnosis (Ascending)')


In [ ]:
# Save outputs
final_out = csv_path.with_name('encounters_ordered_pipeline.csv')
final_df.to_csv(final_out, index=False)

ad_src_out = csv_path.with_name('encounters_by_admissionsource_summary.csv')
ad_typ_out = csv_path.with_name('encounters_by_admissiontype_summary.csv')
admission_source_summary.to_csv(ad_src_out, index=False)
admission_type_summary.to_csv(ad_typ_out, index=False)

print(f"\n{'='*60}")
print("  OUTPUT FILES SAVED")
print('='*60)
print(f"\n  Main output: {final_out.name}")
print(f"  Admission Source summary: {ad_src_out.name}")
print(f"  Admission Type summary: {ad_typ_out.name}")
